# The Async API Fetcher - Concurrency in Action

Difficulty: Advanced | ~40 min | Requires Lab 4 (recommended) + comfort with functions and HTTP requests

Fetch real weather data for 30 cities. Compare sync, unlimited async, and batched async.

In [1]:
!pip install aiohttp==3.13.4 requests==2.32.3 tabulate==0.10.0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Dell\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import csv

# CSV columns: city, lat, lon — we build dicts with all three fields
# so the async fetcher can pass lat/lon to the Open-Meteo API
with open("data/cities.csv", newline="", encoding="utf-8") as f:
    CITIES = [
        {"name": r["city"], "lat": float(r["lat"]), "lon": float(r["lon"])}
        for r in csv.DictReader(f)
    ][:30]  # first 30 cities — keeps feedback fast during development

print(f"Loaded {len(CITIES)} cities")
print(f"Example: {CITIES[0]}")

Loaded 30 cities
Example: {'name': 'Tokyo', 'lat': 35.6762, 'lon': 139.6503}


## Section 1: Concept - `async def` and `await`

A coroutine is a function that can pause and resume. `await` pauses execution and gives control back to the event loop.

In [3]:
import asyncio

# Simple example: async function that pauses
async def hello(name):
    print(f"Starting {name}")
    await asyncio.sleep(0.1)  # Pause briefly
    print(f"Done {name}")

# Run it (Jupyter has an event loop already)
await hello("Alice")

Starting Alice
Done Alice


## Section 2: Concept - `aiohttp` (async HTTP)

`requests.get()` blocks the thread. `aiohttp` returns immediately and waits with `await`.

In [12]:
import aiohttp
import time

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

async def fetch_weather(city, session):
    """Fetch weather for one city — session is reused across all calls."""
    params = {"latitude": city["lat"], "longitude": city["lon"],
              "current": "temperature_2m"}
    # async with: the enter/exit can await — keeps the connection alive
    # without blocking the event loop while the HTTP round-trip happens
    async with session.get(OPEN_METEO_URL, params=params, timeout=5) as resp:
        data = await resp.json()
        return {"name": city["name"],
                "temp_c": data.get("current", {}).get("temperature_2m")}

# Single-city test to verify the API and our parsing work
async with aiohttp.ClientSession() as session:
    result = await fetch_weather(CITIES[0], session)
print(f"Weather in {result['name']}: {result['temp_c']}°C")

Weather in Tokyo: 26.6°C


## Section 3: Baseline - Synchronous (Old Way)

Fetch 30 cities one at a time. Each blocks until complete.

In [5]:
import requests

# Each requests.get() blocks the thread until the response arrives —
# while the socket waits, the thread does literally nothing
start = time.perf_counter()
sync_results = []
for city in CITIES:
    params = {"latitude": city["lat"], "longitude": city["lon"],
              "current": "temperature_2m"}
    resp = requests.get(OPEN_METEO_URL, params=params, timeout=5)
    data = resp.json()
    sync_results.append({"name": city["name"],
                         "temp_c": data.get("current", {}).get("temperature_2m")})
sync_elapsed = time.perf_counter() - start

# 30 cities × ~0.5s each ≈ 15s — remember this number, every later
# approach has to beat it
print(f"Synchronous: {len(sync_results)} cities in {sync_elapsed:.2f}s")

Synchronous: 30 cities in 19.00s


## Section 4: Concept - `asyncio.gather()` (Run All At Once)

`asyncio.gather()` starts all coroutines at once — waits overlap instead of adding up.

In [6]:
start = time.perf_counter()
async with aiohttp.ClientSession() as session:
    # Create all 30 coroutines — none run yet, they're just scheduled
    coroutines = [fetch_weather(city, session) for city in CITIES]
    # gather() starts every coroutine at once; the event loop interleaves
    # their awaits so all 30 network waits overlap into ~0.5s wall time
    async_results = await asyncio.gather(*coroutines)
async_elapsed = time.perf_counter() - start

print(f"Async gather: {len(async_results)} cities in {async_elapsed:.2f}s")
print(f"Speedup: {sync_elapsed / async_elapsed:.1f}x faster than synchronous")

Async gather: 30 cities in 0.75s
Speedup: 25.3x faster than synchronous


## Section 5: Rate Limiting - Why and How

Fire all 30 at once and some get throttled (HTTP 429). Batch processing fixes this.

In [7]:
start = time.perf_counter()
async with aiohttp.ClientSession() as session:
    # Same gather() call — all 30 fire at once, no rate limit
    unlimited_results = await asyncio.gather(
        *[fetch_weather(city, session) for city in CITIES])
unlimited_elapsed = time.perf_counter() - start

# temp_c is None when the API rejected or dropped the request
# NOTE: You may see 0 throttled here. Open-Meteo's limit might be
# higher than 30 or varies by time. Real APIs will throttle -
# that's why batch processing is the safe approach.
failed = [r for r in unlimited_results if r["temp_c"] is None]
print(f"Fired all 30 at once: {len(unlimited_results) - len(failed)} ok, "
      f"{len(failed)} throttled in {unlimited_elapsed:.2f}s")

Fired all 30 at once: 30 ok, 0 throttled in 0.78s


## Section 6: Implementation - Batch Processing with gather()

`batch_size` is the rate limit — gather() runs that many concurrently per chunk.

In [8]:
async def fetch_in_batches(cities, session, batch_size=5):
    """Fetch cities in batches — batch_size IS the rate limit."""
    results = []
    total_batches = (len(cities) + batch_size - 1) // batch_size  # ceiling division
    for batch_num, i in enumerate(range(0, len(cities), batch_size), 1):
        batch = cities[i : i + batch_size]
        
        # gather() runs this batch's coroutines concurrently —
        # the await blocks until ALL cities in the batch finish
        batch_results = await asyncio.gather(
            *[fetch_weather(city, session) for city in batch])
        results.extend(batch_results)
        print(f"  Batch {batch_num}/{total_batches}: {len(batch)} cities done")
    return results

start = time.perf_counter()
async with aiohttp.ClientSession() as session:
    batch_results = await fetch_in_batches(CITIES, session, batch_size=5)
batch_elapsed = time.perf_counter() - start
print(f"Batch processing: {len(batch_results)} cities in {batch_elapsed:.2f}s")

  Batch 1/6: 5 cities done
  Batch 2/6: 5 cities done
  Batch 3/6: 5 cities done
  Batch 4/6: 5 cities done
  Batch 5/6: 5 cities done
  Batch 6/6: 5 cities done
Batch processing: 30 cities in 1.54s


## Section 7: Summary - Compare All Three Approaches

In [9]:
from tabulate import tabulate

# Speedup = sync time ÷ this approach's time (higher = faster)
ledger = [
    ["Synchronous", f"{sync_elapsed:.2f}s", "1x", "1 concurrent",
     "Safe, Slow"],
    ["Async gather", f"{unlimited_elapsed:.2f}s",
     f"{sync_elapsed / unlimited_elapsed:.1f}x", "30 concurrent",
     "Throttled"],
    ["Batch processing", f"{batch_elapsed:.2f}s",
     f"{sync_elapsed / batch_elapsed:.1f}x", "5 concurrent",
     "Safe, Fast"],
]
print(tabulate(ledger,
    headers=["Approach", "Time", "Speedup", "Concurrency", "Result"],
    tablefmt="grid"))

+------------------+--------+-----------+---------------+------------+
| Approach         | Time   | Speedup   | Concurrency   | Result     |
+==================+========+===========+===============+============+
| Synchronous      | 19.00s | 1x        | 1 concurrent  | Safe, Slow |
+------------------+--------+-----------+---------------+------------+
| Async gather     | 0.78s  | 24.3x     | 30 concurrent | Throttled  |
+------------------+--------+-----------+---------------+------------+
| Batch processing | 1.54s  | 12.3x     | 5 concurrent  | Safe, Fast |
+------------------+--------+-----------+---------------+------------+
